# Waste classifier: evaluation & tuning

**Part 1** scores the trained model in `models/` on the held-out TEST set: headline metrics, per-class report, confusion matrix, ROC/PR curves, calibration, decision thresholds, the app's "Not sure" cutoff, and the mistakes it makes.

**Part 2** tunes hyperparameters:
- **Stage A:** a fast grid over the classifier head (dropout, hidden units, learning rate) trained on cached MobileNetV2 features. Each trial takes seconds.
- **Stage B:** full-image trials for the settings that can't use cached features: data augmentation and fine-tuning the top of the backbone.

**Rule used throughout:** every choice (thresholds, cutoffs, hyperparameters) is made on the **validation** split, a 10% slice of TRAIN. TEST is only used to report scores, so they stay unbiased.

Run from the repo's `.venv`: `pip install -r requirements-dev.txt`, then open this notebook with that kernel.

In [ ]:
# ---- Settings ----
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

MODEL_PATH = ROOT / "models" / "waste_classifier.keras"
BATCH_SIZE = 64
POSITIVE = "R"                  # class treated as "positive" for ROC/PR/threshold metrics
TARGET_SURE_ACCURACY = 0.97     # pick the "Not sure" cutoff so confident answers hit this accuracy

RUN_TUNING = True
FEATURE_CACHE = ROOT / "models" / "cache" / "mobilenet_features.npz"
HEAD_MAX_EPOCHS = 40            # Stage A (early stopping usually ends sooner)
STAGE_B_SUBSET = 0.3            # fraction of training batches per epoch in Stage B trials
STAGE_B_HEAD_EPOCHS = 3
STAGE_B_FINE_TUNE_EPOCHS = 3
FINAL_EPOCHS = 10               # final retrain on all of TRAIN (early stopping applies)
FINAL_FINE_TUNE_EPOCHS = 5
SAVE_BEST = False               # True: retrain the winner on all of TRAIN and save models/waste_classifier_tuned.keras

In [ ]:
import os, time, json, warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn import metrics as skm

from waste_classifier import evaluation as ev
from waste_classifier.model import build_mobilenet, build_feature_extractor, build_head
from waste_classifier.train import load_data, fit

keras.utils.set_random_seed(42)
warnings.filterwarnings("ignore", message=".*shuffle=True. was passed.*")
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
BIN_COLOURS = {"O": "#7b5132", "R": "#1f5aa6"}   # brown bin / blue bin, as in the app

In [ ]:
train_ds, val_ds, test_ds = load_data(batch_size=BATCH_SIZE)
CLASS_NAMES = train_ds.class_names
POS = CLASS_NAMES.index(POSITIVE)

def counts(ds):
    y = np.concatenate([np.argmax(l, 1) for _, l in ds])
    return {c: int((y == i).sum()) for i, c in enumerate(CLASS_NAMES)}

split_counts = pd.DataFrame({"train": counts(train_ds), "val": counts(val_ds), "test": counts(test_ds)}).T
split_counts.assign(total=split_counts.sum(axis=1), share_R=split_counts["R"] / split_counts.sum(axis=1))

## Part 1: Score the current model

Predictions on validation (used to choose thresholds) and test (used to report).

In [ ]:
model = keras.models.load_model(MODEL_PATH)
print(f"{model.name} from {MODEL_PATH.name}, {model.count_params():,} params")

t0 = time.time()
y_val, p_val = ev.predict_dataset(model, val_ds)
y_test, p_test = ev.predict_dataset(model, test_ds)
print(f"predicted {len(y_val) + len(y_test):,} images in {time.time() - t0:.0f}s")

### Headline metrics

- **accuracy / balanced accuracy:** balanced accuracy averages per-class recall, so it isn't flattered by the O-heavy class split.
- **macro F1, MCC:** single-number summaries that hold up under class imbalance. For MCC, 0 is chance and 1 is perfect.
- **ROC AUC / average precision:** ranking quality, independent of any threshold.
- **log loss, Brier, ECE:** quality of the probabilities themselves. ECE is the average gap between stated confidence and actual accuracy.

In [ ]:
pd.DataFrame({"validation": ev.summary(y_val, p_val, CLASS_NAMES, POSITIVE),
              "test": ev.summary(y_test, p_test, CLASS_NAMES, POSITIVE)})

In [ ]:
ev.per_class_report(y_test, p_test, CLASS_NAMES)

In [ ]:
cm = skm.confusion_matrix(y_test, p_test.argmax(1))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, data, title, fmt in [(axes[0], cm, "Counts", "d"),
                             (axes[1], cm / cm.sum(1, keepdims=True), "Row-normalised (recall)", ".1%")]:
    ax.imshow(data, cmap="Greys", vmin=0)
    for (i, j), v in np.ndenumerate(data):
        ax.text(j, i, format(v, fmt), ha="center", va="center",
                color="white" if data[i, j] > data.max() / 2 else "black", fontsize=12)
    ax.set(xticks=range(len(CLASS_NAMES)), yticks=range(len(CLASS_NAMES)),
           xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, xlabel="predicted", ylabel="actual", title=title)
plt.tight_layout()

In [ ]:
y_bin = (y_test == POS).astype(int)
fpr, tpr, _ = skm.roc_curve(y_bin, p_test[:, POS])
prec, rec, _ = skm.precision_recall_curve(y_bin, p_test[:, POS])

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.8))
a.plot(fpr, tpr, color=BIN_COLOURS[POSITIVE]); a.plot([0, 1], [0, 1], ls=":", color="grey")
a.set(xlabel="false positive rate", ylabel="true positive rate",
      title=f"ROC ({POSITIVE} positive), AUC {skm.auc(fpr, tpr):.3f}")
b.plot(rec, prec, color=BIN_COLOURS[POSITIVE]); b.axhline(y_bin.mean(), ls=":", color="grey")
b.set(xlabel="recall", ylabel="precision",
      title=f"Precision-recall, AP {skm.average_precision_score(y_bin, p_test[:, POS]):.3f}")
plt.tight_layout()

### Calibration

Is "90% confident" right 90% of the time? Points below the diagonal mean the model is over-confident. The histogram shows how confidence differs between correct and wrong answers, which is what makes a "Not sure" cutoff useful.

In [ ]:
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(y_bin, p_test[:, POS], n_bins=10, strategy="quantile")
conf = p_test.max(1); correct = p_test.argmax(1) == y_test

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.8))
a.plot([0, 1], [0, 1], ls=":", color="grey")
a.plot(mean_pred, frac_pos, "o-", color=BIN_COLOURS[POSITIVE])
a.set(xlabel=f"predicted P({POSITIVE})", ylabel=f"observed share {POSITIVE}", title="Reliability diagram")
bins = np.linspace(0.5, 1, 21)
b.hist(conf[correct], bins=bins, alpha=.7, label=f"correct ({correct.sum()})", color="#5c665f")
b.hist(conf[~correct], bins=bins, alpha=.8, label=f"wrong ({(~correct).sum()})", color="#a33a2a")
b.set(xlabel="confidence (max probability)", ylabel="images", title="Confidence by outcome", yscale="log")
b.legend(); plt.tight_layout()

### Decision threshold

By default the model picks whichever class has probability above 0.5. If one kind of mistake is worse, the threshold can move. For example, putting organic waste in the blue bin contaminates recycling. The threshold is chosen on validation by balanced accuracy, then applied to test.

In [ ]:
sweep_val = ev.threshold_sweep(y_val, p_val[:, POS], POS)
best_t = float(sweep_val.loc[sweep_val.balanced_accuracy.idxmax(), "threshold"])

ax = sweep_val.plot(x="threshold", y=["precision", "recall", "f1", "balanced_accuracy"], figsize=(7, 3.6),
                    title=f"Validation: predict {POSITIVE} when P({POSITIVE}) >= threshold")
ax.axvline(best_t, ls=":", color="black"); ax.set_ylim(0.5, 1.0)

def at_threshold(y, p, t):
    pred = np.where(p[:, POS] >= t, POS, 1 - POS)
    return {"accuracy": skm.accuracy_score(y, pred), "balanced_accuracy": skm.balanced_accuracy_score(y, pred),
            f"precision_{POSITIVE}": skm.precision_score(y == POS, pred == POS),
            f"recall_{POSITIVE}": skm.recall_score(y == POS, pred == POS)}

pd.DataFrame({"test @ 0.50": at_threshold(y_test, p_test, 0.5),
              f"test @ {best_t:.2f} (chosen on val)": at_threshold(y_test, p_test, best_t)})

### The app's "Not sure" cutoff

The app shows "Not sure, check by hand" when confidence is below `UNSURE_BELOW` in `app/static/app.js` (currently 0.75). A higher cutoff gives more accurate confident answers, but flags more items as unsure.

In [ ]:
cov_val = ev.coverage_curve(y_val, p_val)
cov_test = ev.coverage_curve(y_test, p_test)
ok = cov_val[cov_val.accuracy_when_sure >= TARGET_SURE_ACCURACY]
recommended = float(ok.unsure_below.min()) if len(ok) else float(cov_val.unsure_below.max())

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(cov_test.unsure_below, cov_test.coverage, label="coverage (answered confidently)", color="#5c665f")
ax.plot(cov_test.unsure_below, cov_test.accuracy_when_sure, label="accuracy when confident", color=BIN_COLOURS["R"])
ax.plot(cov_test.unsure_below, cov_test.errors_caught, label="share of errors flagged unsure", color="#a33a2a")
ax.axvline(0.75, ls=":", color="grey"); ax.text(0.752, 0.02, "current 0.75", color="grey")
ax.axvline(recommended, ls="--", color="black"); ax.text(recommended + .002, 0.1, f"recommended {recommended:.3f}")
ax.set(xlabel="UNSURE_BELOW", ylim=(0, 1.01), title="Test set: unsure-cutoff trade-off"); ax.legend(loc="lower left")

print(f"Cutoff with >= {TARGET_SURE_ACCURACY:.0%} confident accuracy on validation: {recommended:.3f}")
cov_test[cov_test.unsure_below.isin([0.5, 0.75, round(recommended, 3)])].set_index("unsure_below")

### Error analysis: the most confident mistakes

These are worth looking at first. They are often mislabelled images or genuinely ambiguous items, such as food in packaging.

In [ ]:
from PIL import Image

paths = np.array(test_ds.file_paths)
wrong = np.where(p_test.argmax(1) != y_test)[0]
worst = wrong[np.argsort(-p_test[wrong].max(1))][:12]

fig, axes = plt.subplots(2, 6, figsize=(14, 5.4))
for ax, i in zip(axes.flat, worst):
    ax.imshow(Image.open(paths[i]).convert("RGB")); ax.axis("off")
    pred = CLASS_NAMES[p_test[i].argmax()]
    ax.set_title(f"true {CLASS_NAMES[y_test[i]]}, said {pred} {p_test[i].max():.0%}\n{Path(paths[i]).name}",
                 fontsize=8, color=BIN_COLOURS[pred])
plt.tight_layout()

## Part 2: Hyperparameter tuning

### Stage A: head grid on cached features

With the backbone frozen, MobileNetV2 turns each image into the same 1280-number feature vector every time. We compute these once, cache them to disk, and train only the small head on them, so each trial takes seconds rather than minutes. The head here is the same `classifier_head` the full model uses, so the settings carry over.

There's no data augmentation in Stage A, because cached features are fixed. Stage B covers that.

In [ ]:
if RUN_TUNING:
    if FEATURE_CACHE.exists():
        cache = np.load(FEATURE_CACHE)
        X_tr, y_tr, X_va, y_va = cache["X_tr"], cache["y_tr"], cache["X_va"], cache["y_va"]
        print(f"loaded cached features from {FEATURE_CACHE.relative_to(ROOT)}")
    else:
        extractor = build_feature_extractor()
        t0 = time.time()
        X_tr, y_tr = ev.extract_features(extractor, train_ds)
        X_va, y_va = ev.extract_features(extractor, val_ds)
        FEATURE_CACHE.parent.mkdir(parents=True, exist_ok=True)
        np.savez(FEATURE_CACHE, X_tr=X_tr, y_tr=y_tr, X_va=X_va, y_va=y_va)
        print(f"extracted features in {time.time() - t0:.0f}s, cached to {FEATURE_CACHE.relative_to(ROOT)}")
    print("train", X_tr.shape, "val", X_va.shape)

In [ ]:
import itertools

GRID = {
    "dropout": [0.0, 0.2, 0.4],
    "hidden_units": [0, 128, 512],
    "learning_rate": [1e-3, 3e-4],
}

def run_head_trial(dropout, hidden_units, learning_rate, batch_size=64):
    keras.utils.set_random_seed(42)
    head = build_head(X_tr.shape[1], len(CLASS_NAMES), dropout, hidden_units, learning_rate)
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    h = head.fit(X_tr, keras.utils.to_categorical(y_tr), batch_size=batch_size, epochs=HEAD_MAX_EPOCHS,
                 validation_data=(X_va, keras.utils.to_categorical(y_va)), callbacks=[stop], verbose=0).history
    best = int(np.argmin(h["val_loss"]))
    p = head.predict(X_va, verbose=0)
    return {"val_loss": h["val_loss"][best], "val_accuracy": h["val_accuracy"][best],
            "val_balanced_accuracy": skm.balanced_accuracy_score(y_va, p.argmax(1)),
            "val_ece": ev.expected_calibration_error(y_va, p), "best_epoch": best + 1}

if RUN_TUNING:
    rows = []
    t0 = time.time()
    for values in itertools.product(*GRID.values()):
        cfg = dict(zip(GRID, values))
        rows.append({**cfg, **run_head_trial(**cfg)})
    stage_a = pd.DataFrame(rows).sort_values("val_loss").reset_index(drop=True)
    print(f"{len(rows)} trials in {time.time() - t0:.0f}s")
    display(stage_a.head(10))

In [ ]:
if RUN_TUNING:
    fig, axes = plt.subplots(1, len(GRID["learning_rate"]), figsize=(4.6 * len(GRID["learning_rate"]), 3.6), sharey=True)
    for ax, lr in zip(np.atleast_1d(axes), GRID["learning_rate"]):
        piv = stage_a[stage_a.learning_rate == lr].pivot(index="dropout", columns="hidden_units", values="val_accuracy")
        im = ax.imshow(piv.values, cmap="Blues", vmin=stage_a.val_accuracy.min(), vmax=stage_a.val_accuracy.max())
        for (i, j), v in np.ndenumerate(piv.values):
            ax.text(j, i, f"{v:.2%}", ha="center", va="center", fontsize=9)
        ax.set(xticks=range(piv.shape[1]), xticklabels=piv.columns, yticks=range(piv.shape[0]),
               yticklabels=piv.index, xlabel="hidden units", ylabel="dropout", title=f"val accuracy, lr={lr:g}")
    plt.tight_layout()
    best_head = stage_a.iloc[0][list(GRID)].to_dict()
    best_head["hidden_units"] = int(best_head["hidden_units"])
    print("best head (lowest val loss):", best_head)

### Stage B: augmentation and fine-tuning

These trials train on real images, so they are slower. Each one trains the best head from Stage A, then optionally unfreezes the top *N* backbone layers at a low learning rate. To keep it quick, each epoch sees a random `STAGE_B_SUBSET` of the training batches, and all trials are compared on the full validation split.

In [ ]:
STAGE_B_TRIALS = [
    {"name": "head only", "augment": False, "fine_tune_layers": 0},
    {"name": "head + augment", "augment": True, "fine_tune_layers": 0},
    {"name": "fine-tune top 30", "augment": False, "fine_tune_layers": 30},
    {"name": "fine-tune top 30 + augment", "augment": True, "fine_tune_layers": 30},
    {"name": "fine-tune top 60", "augment": False, "fine_tune_layers": 60},
]

def run_image_trial(trial, train_data, head_epochs, ft_epochs):
    keras.utils.set_random_seed(42)
    m = build_mobilenet(len(CLASS_NAMES), augment=trial["augment"], **best_head)
    h = fit(m, train_data, val_ds, head_epochs, patience=2, fine_tune_layers=trial["fine_tune_layers"],
            fine_tune_epochs=ft_epochs if trial["fine_tune_layers"] else 0, verbose=0)
    y, p = ev.predict_dataset(m, val_ds)
    return m, h, {"val_loss": skm.log_loss(y, p), "val_accuracy": skm.accuracy_score(y, p.argmax(1)),
                  "val_balanced_accuracy": skm.balanced_accuracy_score(y, p.argmax(1)),
                  "val_ece": ev.expected_calibration_error(y, p), "epochs": len(h["loss"])}

if RUN_TUNING:
    n_batches = int(len(train_ds) * STAGE_B_SUBSET)
    subset = train_ds.take(n_batches)   # train_ds reshuffles each epoch, so each epoch sees a different subset
    rows, histories = [], {}
    for trial in STAGE_B_TRIALS:
        t0 = time.time()
        _, h, scores = run_image_trial(trial, subset, STAGE_B_HEAD_EPOCHS, STAGE_B_FINE_TUNE_EPOCHS)
        histories[trial["name"]] = h
        rows.append({**trial, **scores, "minutes": (time.time() - t0) / 60})
        print(f"{trial['name']:28} val acc {scores['val_accuracy']:.4f}  ({rows[-1]['minutes']:.1f} min)")
    stage_b = pd.DataFrame(rows).sort_values("val_loss").reset_index(drop=True)
    display(stage_b)

In [ ]:
if RUN_TUNING:
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8))
    for name, h in histories.items():
        line, = a.plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], marker="o", ms=3, label=name)
        b.plot(range(1, len(h["val_accuracy"]) + 1), h["val_accuracy"], marker="o", ms=3, color=line.get_color())
    a.axvline(STAGE_B_HEAD_EPOCHS + .5, ls=":", color="grey"); b.axvline(STAGE_B_HEAD_EPOCHS + .5, ls=":", color="grey")
    a.set(xlabel="epoch", ylabel="val loss", title="Validation loss (dotted line: fine-tuning starts)")
    b.set(xlabel="epoch", ylabel="val accuracy", title="Validation accuracy")
    a.legend(fontsize=8); plt.tight_layout()

### Result

The winning configuration as a training command. Stage B used a subset of the data and few epochs, so for the final model train on all of TRAIN with more epochs (early stopping still applies).

In [ ]:
if RUN_TUNING:
    win = stage_b.iloc[0]
    cmd = [f"python -m waste_classifier.train --arch mobilenet --epochs {FINAL_EPOCHS}",
           f"--dropout {best_head['dropout']:g} --hidden-units {best_head['hidden_units']} "
           f"--learning-rate {best_head['learning_rate']:g}"]
    if win.augment:
        cmd.append("--augment")
    if win.fine_tune_layers:
        cmd.append(f"--fine-tune-layers {int(win.fine_tune_layers)} --fine-tune-epochs {FINAL_FINE_TUNE_EPOCHS}")
    print("Best Stage B trial:", win["name"])
    print("\n" + " \\\n    ".join(cmd))

In [ ]:
if RUN_TUNING and SAVE_BEST:
    keras.utils.set_random_seed(42)
    final = build_mobilenet(len(CLASS_NAMES), augment=bool(win.augment), **best_head)
    fit(final, train_ds, val_ds, epochs=FINAL_EPOCHS, patience=3,
        fine_tune_layers=int(win.fine_tune_layers), fine_tune_epochs=FINAL_FINE_TUNE_EPOCHS if win.fine_tune_layers else 0)
    y_t, p_t = ev.predict_dataset(final, test_ds)
    display(pd.DataFrame({"current model": ev.summary(y_test, p_test, CLASS_NAMES, POSITIVE),
                          "tuned model": ev.summary(y_t, p_t, CLASS_NAMES, POSITIVE)}))
    out = ROOT / "models" / "waste_classifier_tuned.keras"
    final.save(out)
    out.with_suffix(".labels.json").write_text(json.dumps(CLASS_NAMES))
    print(f"saved {out.relative_to(ROOT)}. To use it in the app: WASTE_MODEL_PATH={out.relative_to(ROOT)} uvicorn app.main:app")